In [1]:
#------------------------------------------------ Begin_Librairie ----------------------------------------

import os
import pandas as pd
from bs4 import BeautifulSoup
import datetime
from selenium import webdriver
from time import sleep
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from io import StringIO

In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'RO ASFR' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now=datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment
os.chdir(scriptfolder)
tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)


Running RO ASFR Web Scraping Tool v.1.1


In [3]:
#------------------------------------------------ Begin_Variable ----------------------------------------
regdict={
         regulatorName+' 1': ['https://data.asfromania.ro/scr/ra?sectiune=1&tipCompanie=0&l=en',  
                              'https://data.asfromania.ro/scr/ra?sectiune=1&tipCompanie=1&l=en'], 

        #  regulatorName+' 2': 'https://data.asfromania.ro/registru/lista.php?listasect=1&lng=1', 

        #  regulatorName+' 3': 'https://data.asfromania.ro/registru/lista.php?listasect=1&lng=1', 

        #  regulatorName+' 4': 'https://data.asfromania.ro/registru/lista.php?listasect=1&lng=1', 

        # regulatorName+' 5': 'https://data.asfromania.ro/registru/lista.php?listasect=1&lng=1',   
        # regulatorName+' 6': 'https://data.asfromania.ro/registru/lista.php?listasect=1&lng=1', 

        # regulatorName+' 7': 'https://data.asfromania.ro/registru/lista.php?listasect=1&lng=1', 

        # regulatorName+' 8': 'https://data.asfromania.ro/registru/lista.php?listasect=1&lng=1', 
        # regulatorName+' 9': 'https://data.asfromania.ro/registru/lista.php?listasect=1&lng=1', 
        }

Reg_Detail = {
        regulatorName+' 2': 'Entitati care presteaza servicii si activitati de investitii in Romania', 
        regulatorName+' 3': 'Societati de administrare a investitiilor', 
        regulatorName+' 4': 'Administratori de fonduri de investitii alternative', 
        regulatorName+' 5': 'Fonduri de investitii alternative', 
        regulatorName+' 6': 'Operatori de piata', 
        regulatorName+' 7': 'Entitati care efectueaza operatiuni posttranzactionare', 
        regulatorName+' 8': 'Distribuitori de titluri de participare ale OPC', 
        regulatorName+' 9': 'Furnizori de servicii de finantare participativa', 

}

Typology ={

        regulatorName+' 1': 'Register of Insurance companies and Main intermediaries',    

        regulatorName+' 2': 'Entities providing investment services and activities in Romania', 

        regulatorName+' 3': 'Investment management companies', 

        regulatorName+' 4': 'Alternative investment fund managers', 

        regulatorName+' 5': 'Alternative investment funds', 

        regulatorName+' 6': 'Market operators', 

        regulatorName+' 7': 'Entities that perform post-trading operations', 

        regulatorName+' 8': 'Distributors of units of Collective Investment Undertakings', 
        regulatorName+' 9': 'Crowdfunding service providers', 

        }



sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

         'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

         'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

         'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

         'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

         'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')

sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')



In [4]:
#------------------------------------------------ Begin_chromedriver ----------------------------------------
#Starting Chrome driver, set to download files in tempfolder
chromeOptions = webdriver.ChromeOptions()
prefs = {"plugins.always_open_pdf_externally": True,
		 "download.prompt_for_download": False,
		 "download.default_directory" : tempfolder,
         'profile.default_content_setting_values.automatic_downloads': 1 # Desable a Multiplefile download alert
         }
chromeOptions.add_experimental_option("prefs",prefs)
chromeOptions.add_argument("--ignore-certificate-errors")
chromeOptions.add_argument("--allow-insecure-localhost")
chromeOptions.add_argument("--disable-web-security")  # optional; often not needed
# driver = webdriver.Chrome(options=chromeOptions)
# driver.maximize_window()

# driver.execute_cdp_cmd("Network.enable", {})
# driver.execute_cdp_cmd("Network.setExtraHTTPHeaders", {
#     "headers": {
#         "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36",
#         "Accept-Language": "en-US,en;q=0.9"
#     }
# })



In [5]:
#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty
    return sqldict


In [6]:
#------------------------------------------------ Begin_Main ----------------------------------------


for index, reg in enumerate(regdict):
    print(f'[Start New Reg] -- Working with list { reg} --')
    driver = webdriver.Chrome(options=chromeOptions)
    driver.maximize_window()

    driver.execute_cdp_cmd("Network.enable", {})
    driver.execute_cdp_cmd("Network.setExtraHTTPHeaders", {
        "headers": {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36",
            "Accept-Language": "en-US,en;q=0.9"
        }
    })

    
    sleep(2)  # wait for the page to load
    if reg == 'RO ASFR 1':
        for link in regdict[reg]:
            driver.get(link)
            sleep(2)
            btn_group = driver.find_element(By.CLASS_NAME, 'dt-buttons')
            sleep(1)
            excel_button = btn_group.find_element(By.CLASS_NAME, 'buttons-excel')
            sleep(1)
            excel_button.click()
            sleep(5)  
            files = os.listdir(tempfolder)
            data_excel = pd.read_excel(tempfolder + '/' + files[0], engine='openpyxl')
            data_excel.columns = data_excel.iloc[0].str.strip()
            data_excel = data_excel[1:] 
            for _, row in data_excel.iterrows():
                name_ = row["Name"]
                address_ = row["Address"]
                enrollment_date = row["Enrollment date"]
                fisal_code = row["Fiscal code"]
                registry_id = row["Registry of Commerce ID"]
                sqldict['Name'].append(name_)
                sqldict['Address_1'].append(address_)
                sqldict['InternalID_1'].append(registry_id) 
                sqldict['InternalID_1_type'].append('Registry of Commerce ID')
                sqldict['InternalID_2'].append(fisal_code)
                sqldict['InternalID_2_type'].append('Fiscal code')
                sqldict['RegulationDate'].append(enrollment_date)
                sqldict['ListProcessDate'].append(processdate)
                sqldict['RegCtry'].append(reg.split()[0])
                sqldict['RegCode'].append(reg.split()[1])
                sqldict['ListCode'].append(reg.split()[2])
                sqldict['RegulationType'].append('Regulated')
                sqldict['ListName'].append(Typology[reg])
                sqldict = bourange_same_length_array(sqldict)
            os.remove(tempfolder + '/' + files[0])

    else:
        driver.get(regdict[reg])
        sleep(2)
        detail = Reg_Detail.get(reg)
        if not detail:
            print(f"Missing Reg_Detail for {reg}")
            continue
        wait = WebDriverWait(driver, 15)
        link = wait.until(
            EC.presence_of_element_located(
                (By.XPATH, f"//a[contains(normalize-space(.), {repr(detail)})]")
            )
        )

        # option A: click
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", link)
        try:
            link.click()
        except Exception:
            driver.execute_script("arguments[0].click();", link)
        sleep(3)
        html = driver.page_source
        tables = pd.read_html(StringIO(html))
        # tables = pd.read_html(driver.current_url)
        # df = tables[1]
        for _, row in tables[1].iterrows():
            register_no = row[tables[1].columns[1]]
            register_date = row[tables[1].columns[2]]
            name_ = row[tables[1].columns[4]]
            # codes_cui_nrorc = row[tables[1].columns[6]]
            city_ =  row[tables[1].columns[9]]
            address_ = row[tables[1].columns[10]]
            tel_ = row[tables[1].columns[11]]
            fax_ = row[tables[1].columns[12]]
            email_ = row[tables[1].columns[13]]
            website_ = row[tables[1].columns[14]]

            sqldict['Name'].append(name_)
            sqldict['Address_1'].append(address_)
            sqldict['City'].append(city_)
            sqldict['InternalID_1'].append(register_no)
            sqldict['InternalID_1_type'].append('NR.REGISTRU')
            # sqldict['InternalID_2'].append(codes_cui_nrorc)
            # sqldict['InternalID_2_type'].append('CODURI (CUI,NRORC,...)')
            sqldict['RegulationDate'].append(register_date)
            sqldict['Phone'].append(tel_)
            sqldict['Fax'].append(fax_)
            sqldict['Email'].append(email_)
            sqldict['Website'].append(website_)
            sqldict['ListProcessDate'].append(processdate)
            sqldict['RegCtry'].append(reg.split()[0])
            sqldict['RegCode'].append(reg.split()[1])
            sqldict['ListCode'].append(reg.split()[2])
            sqldict['RegulationType'].append('Regulated')
            sqldict['ListName'].append(Typology[reg])
            sqldict = bourange_same_length_array(sqldict)
    driver.quit()




[Start New Reg] -- Working with list RO ASFR 1 --


In [7]:
for link in regdict[reg]:
    print(link)

https://data.asfromania.ro/scr/ra?sectiune=1&tipCompanie=0&l=en
https://data.asfromania.ro/scr/ra?sectiune=1&tipCompanie=1&l=en


In [8]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(filename, index=False)
sleep(3)